# BSCP sack-train-ml — Compile YOLOv11s → HEF (Colab · DFC ClientRunner)

> Loom Oracle (AI) · 2026-06-23 · คู่กับ `train_run.ipynb` (อันนั้นเทรน, อันนี้ compile)
> pipeline (ตรงกับ DFC API): **`.onnx → parse(.har) → optimize/quantize+calibrate(.har) → compile(.hef)`**

## ▶ เริ่ม: Runtime → Change runtime type → **High-RAM** (GPU ถ้ามี ช่วย optimize เร็วขึ้น) → รันทีละ section

---
### ⚠️ เตือน 1 — เวอร์ชันต้องโหลดบน edge ได้
edge รัน **HailoRT 4.20.0** → HEF ต้องโหลดบน 4.20.0 ได้
- ชัวร์สุด: **DFC 3.30.0 + model-zoo 2.14.0** (Hailo Suite 2025-01)
- ถ้าใช้ DFC ใหม่กว่า (3.33.x) → **ต้องทดสอบโหลดบน Pi จริง** (section 11) ถ้า error `invalid compiled format` ค่อยถอยมา 3.30

### ✅ เตือน 2 — โมเดล custom 2 คลาส (verified โดย Loom)
best.onnx นี้ถูกตรวจแล้ว: detect head `model.23`, cls conv = **2 channels** (Person, Sack), box conv = 64 (4×DFL16).
6 end-nodes + input name (`images`) ใส่ไว้ใน config ด้านล่างจาก onnx จริง — ไม่ต้องเดา


## 0) Config — แก้ค่าตรงนี้ที่เดียว


In [ ]:
# ===== config =====
HW_ARCH      = 'hailo8l'
NET_NAME     = 'yolov11s_sack'   # ต้องตรงกับขนาดโมเดลที่เทรนจริง (s/m/l) —
                                 # ค่านี้ไปเป็นชื่อไฟล์ .hef และ 'model' ใน meta.yaml
NUM_CLASSES  = 2
CLASS_NAMES  = ['Person', 'Sack']
INPUT_SIZE   = 640
START_NODE   = 'images'                 # verified จาก onnx

# 6 end-nodes (verified จาก best.onnx) — จุดตัดก่อน DFL+sigmoid ให้ NMS บนชิปทำต่อ
END_NODES = [
    '/model.23/cv2.0/cv2.0.2/Conv', '/model.23/cv3.0/cv3.0.2/Conv',   # stride 8  (box64, cls2)
    '/model.23/cv2.1/cv2.1.2/Conv', '/model.23/cv3.1/cv3.1.2/Conv',   # stride 16
    '/model.23/cv2.2/cv2.2.2/Conv', '/model.23/cv3.2/cv3.2.2/Conv',   # stride 32
]

# NMS contract (จาก edge hailo_backend.py analysis)
NMS_SCORES_TH = 0.20    # <=0.25 กัน flagging band [0.35,0.70) หาย
NMS_IOU_TH    = 0.70    # สูงไว้ กันสองกระสอบติดกันโดน merge เป็นอันเดียว
MAX_PER_CLASS = 50      # = --max_det ของ edge
REG_LENGTH    = 16      # DFL (box conv 64 = 4*16)

# paths — ทำงานใน MyDrive/hailo/ ที่เดียว (ไม่ copy ออกมา /content)
HAILO_DIR = '/content/drive/MyDrive/hailo'   # ใส่ DFC whl + best.onnx + calib_v34.zip ไว้ที่นี่
ONNX_PATH = f'{HAILO_DIR}/best.onnx'
CALIB_DIR = f'{HAILO_DIR}/calib'             # แตกจาก calib_v34.zip (section 1)
WORK      = f'{HAILO_DIR}/work'              # .har/.hef เขียนลง Drive -> persist ไม่ต้อง download
OUT_HEF   = f'{WORK}/{NET_NAME}.hef'
CALIB_N   = 512                              # 256-1024; มากขึ้น = quantize นิ่งขึ้น แต่ช้าลง

SOURCE_COMMIT = 'cb0c3ab'   # sack-train-ml sha ตอน export onnx

# NOTE: WORK อยู่บน Drive -> makedirs ทำหลัง mount (section 1)
print('config ok:', NET_NAME, NUM_CLASSES, 'classes ->', HW_ARCH, '| end_nodes:', len(END_NODES))


## 0.5) Sync repo — อ่าน SHA ที่พิมพ์ออกมาทุกครั้ง

Colab **cache ตัว notebook** ไว้: เปิดจาก GitHub ทิ้งไว้แล้วกด Run all จะได้ cell ชุดเก่า
ไม่ใช่ของบน `main` เคยเสียเวลาไปหนึ่งรอบเพราะเรื่องนี้ — แก้บั๊กแล้ว push แล้ว แต่ที่รันคือฉบับเก่า

cell นี้จึงดึงโค้ดจริงจาก `main` มาเอง สคริปต์ compile ที่ใช้จะสดเสมอแม้ notebook จะเก่า
และ SHA ที่พิมพ์คือหลักฐานว่ากำลังรันอะไรอยู่ — เทียบกับ commit ล่าสุดบน GitHub ได้เลย

ถ้า **ตัว notebook เอง** เก่า (เช่น cell นี้ไม่มี) ให้ปิดแท็บ เปิด URL ใหม่ แล้ว
Runtime → Disconnect and delete runtime ก่อน Run all


In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/pitikorn-pam/sack-train-ml.git'
REPO_DIR = Path('/content/sack-train-ml')
BRANCH   = 'main'

if REPO_DIR.exists():
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

GIT_SHA = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                  cwd=REPO_DIR, text=True).strip()

# COMPILE_COMMIT, deliberately NOT SOURCE_COMMIT. The config cell's SOURCE_COMMIT is
# the sha the ONNX was exported at; this is the sha doing the compiling. Folding one
# into the other would make meta.yaml claim the model came from code that only ever
# quantized it — and the point of these fields is to answer "which code made this?"
COMPILE_COMMIT = GIT_SHA
COMPILE_SCRIPT = REPO_DIR / 'scripts' / 'compile_clientrunner.py'
assert COMPILE_SCRIPT.exists(), f'compile script missing in the clone: {COMPILE_SCRIPT}'

print('repo synced at', GIT_SHA)
print('compile script:', COMPILE_SCRIPT)

## 1) Mount Drive + เตรียมไฟล์ (ทำงานใน `MyDrive/hailo/` เท่านั้น)
ใส่ 3 ไฟล์ไว้ใน `MyDrive/hailo/` ก่อน:
- **DFC whl** `hailo_dataflow_compiler-3.x-...whl` (Hailo Developer Zone, gated)
- **best.onnx**
- **calib_v34.zip** (≥256 รูปจริงจาก edge — Loom เตรียม 512 รูปให้แล้ว)

cell นี้ mount + makedirs(WORK) + แตก calib ลง Drive + verify — **ไม่ copy ออกมา /content**


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import glob, os
os.makedirs(WORK, exist_ok=True)
# แตก calib ลง Drive (ครั้งแรกเท่านั้น)
if len(glob.glob(f'{CALIB_DIR}/*.jpg')) < 256:
    !unzip -q -o {HAILO_DIR}/calib_v34.zip -d {CALIB_DIR}
DFC_WHL = glob.glob(f'{HAILO_DIR}/hailo_dataflow_compiler*.whl')
print('HAILO_DIR:', HAILO_DIR)
print('onnx     :', os.path.exists(ONNX_PATH), os.path.getsize(ONNX_PATH) if os.path.exists(ONNX_PATH) else '-')
print('calib n  :', len(glob.glob(f'{CALIB_DIR}/*.jpg')), '(ควร 512)')
print('DFC whl  :', DFC_WHL)
assert DFC_WHL, '❌ ไม่เจอ DFC wheel ใน MyDrive/hailo/'
assert os.path.exists(ONNX_PATH), '❌ ไม่เจอ best.onnx ใน MyDrive/hailo/'
assert len(glob.glob(f'{CALIB_DIR}/*.jpg')) >= 256, '❌ calib < 256 (เช็ค calib_v34.zip บน Drive)'
print('✅ ครบ 3 ไฟล์ พร้อม compile')


## 2) Runtime check (High-RAM)
compile กิน RAM 12–32GB, ใช้ CPU เป็นหลัก (GPU ช่วย optimize)


In [ ]:
import subprocess, sys
print(subprocess.run(['free','-h'],capture_output=True,text=True).stdout)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout or 'no GPU (CPU-only, ช้าแต่ได้)')
print('python', sys.version.split()[0], '(DFC ต้องการ 3.10/3.11)')


## 3) System deps + pins
Colab cuDNN 9 ทำ DFC พัง — pin numpy/scipy ที่ DFC ต้องการ


In [ ]:
!sudo apt-get -qq update
!sudo apt-get -qq install -y python3-dev python3-tk libfuse2 graphviz libgraphviz-dev >/dev/null 2>&1
print('apt deps ok')


## 4) venv + ติดตั้ง DFC (Python 3.11 — บังคับ)

DFC อยู่ใน virtualenv แยก (กัน dep ชนกับ kernel). ClientRunner เรียกผ่าน `!hailo_venv/bin/python script.py`

**ต้องเป็น Python 3.11 หรือ 3.10 เท่านั้น** ไม่ใช่ `python3` ของ Colab ซึ่งตอนนี้เป็น 3.13:

| package | มี wheel ให้ python |
|---|---|
| `numpy==1.23.3` | 3.8–**3.11** |
| `scipy==1.10.1` | 3.8–**3.11** |
| `scipy==1.12.0` (DFC pin เอง) | 3.9–3.12 |

บน 3.12+ ทั้งสามตัวไม่มี wheel → pip ไปสร้างจาก sdist → ล้มด้วย
`metadata-generation-failed` แล้ว `hailo_sdk_client` ก็ไม่เคยถูกติดตั้ง

ใช้ `uv` ดึง CPython 3.11 แบบ standalone — ไม่ต้องพึ่ง apt/PPA ของ distro
และ `--seed` ให้ **setuptools** มาด้วย ซึ่งจำเป็น เพราะ numpy 1.23.3 ต้อง
`setuptools.build_meta` ถ้าไม่มีจะได้ `Cannot import 'setuptools.build_meta'`


In [ ]:
# venv บน Python 3.11 — ไม่ใช่ python3 ของระบบ (Colab = 3.13 ซึ่ง DFC ใช้ไม่ได้)
import subprocess, sys
from pathlib import Path

VENV_DIR = '/content/hailo_venv'
VENV = f'{VENV_DIR}/bin'

!pip -q install uv
!uv python install 3.11
!rm -rf {VENV_DIR}
!uv venv --python 3.11 --seed {VENV_DIR}

# ยืนยันเวอร์ชันจาก venv จริง ก่อนลงอะไรต่อ
#
# ห้ามใช้ `!` magic ตรวจเวอร์ชัน: IPython ขยาย {...} เป็นตัวแปรของ kernel ก่อนส่งเข้า
# shell ดังนั้น `!{VENV}/python -c "print(f'{sys.version_info.major}...')"` จะถูกแทนค่า
# ด้วยเวอร์ชันของ kernel ตั้งแต่ก่อนรัน — คำสั่งที่ออกไปกลายเป็น print(f'3.13') ซึ่ง
# พิมพ์ 3.13 ไม่ว่า venv จะเป็นอะไร นั่นทำให้ assert ฟ้องผิดทั้งที่ venv เป็น 3.11 ถูกแล้ว
# subprocess ไม่มี brace expansion จึงวัดของจริง
probe = subprocess.run(
    [f'{VENV}/python', '-c', 'import sys; print("%d.%d" % sys.version_info[:2])'],
    capture_output=True, text=True)
PYVER = probe.stdout.strip()
print('venv python:', PYVER or f'(probe failed: {probe.stderr.strip()[:200]})')
assert probe.returncode == 0, f'เรียก {VENV}/python ไม่ได้ — venv ไม่ถูกสร้าง?'
assert PYVER in ('3.10', '3.11'), (
    f'venv เป็น python {PYVER} แต่ DFC ต้อง 3.10/3.11 — numpy==1.23.3 ไม่มี wheel '
    f'สำหรับ {PYVER} และ build จาก sdist ไม่ผ่าน'
)

!{VENV}/pip -q install --upgrade pip setuptools wheel
!{VENV}/pip install numpy==1.23.3 scipy==1.10.1 pillow onnx
!{VENV}/pip install {DFC_WHL[0]}    # wheel จาก MyDrive/hailo/
# model-zoo optional (eval เท่านั้น):
# !{VENV}/pip install {HAILO_DIR}/hailo_model_zoo*.whl

In [ ]:
# verify DFC พร้อม + เวอร์ชัน — ต้องผ่านก่อนไป cell ถัดไป
import subprocess
r = subprocess.run(
    ['/content/hailo_venv/bin/python', '-c',
     "from hailo_sdk_client import ClientRunner; import hailo_sdk_client as c; "
     "print('hailo_sdk_client', getattr(c,'__version__','?'))"],
    capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, (
    'DFC ยังไม่ได้ติดตั้งใน venv — อย่าไป compile ต่อ เพราะ cell 7 จะล้มแบบเงียบ '
    'แล้ว cell ถัดไปจะไปหยิบ .hef เก่ามารายงานว่าสำเร็จ'
)

## 5) Calibration set
คุณภาพ calib = ตัวชี้เป็นชี้ตายของ recall หลัง quantize (โดยเฉพาะคลาสน้อย = Person).
ใช้ **รูปจริงจาก edge** ครอบคลุม กลางวัน/กลางคืน/ฝุ่น/บัง/มุมกล้องจริง ≥256 รูป.
compile script จะ **letterbox 640 (pad 114) แบบเดียวกับ edge** แล้วป้อนเป็น 0–255 (normalization layer หาร 255 บนชิป)


In [ ]:
# manifest + dataset hash (ไปใส่ meta.yaml)
import glob, hashlib, csv
imgs = sorted(glob.glob(f'{CALIB_DIR}/*.jpg')+glob.glob(f'{CALIB_DIR}/*.jpeg')+glob.glob(f'{CALIB_DIR}/*.png'))
assert len(imgs) >= 256, f'ต้องการ >=256 รูป มี {len(imgs)} — เพิ่ม calib ก่อน'
h = hashlib.sha256()
with open(f'{WORK}/calibration_manifest.csv','w',newline='') as f:
    w = csv.writer(f)
    for p in imgs:
        d = hashlib.sha256(open(p,'rb').read()).hexdigest(); h.update(d.encode()); w.writerow([p,d])
DATASET_HASH = h.hexdigest()
print('calib:', len(imgs), '| dataset_hash:', DATASET_HASH[:16], '...')


## 6) compile script — มาจาก repo ไม่ใช่เขียนใหม่ที่นี่

ก่อนหน้านี้ cell นี้ `%%writefile` สคริปต์ฉบับย่อ 85 บรรทัดของตัวเอง ขณะที่
`scripts/compile_clientrunner.py` ใน repo มี 263 บรรทัด สองฉบับแยกทางกันไปแล้ว —
ฉบับใน repo auto-derive end-nodes จาก ONNX graph (รองรับ v8/v9/v12/YOLO26 ที่ head
ไม่ได้อยู่ที่ `model.23`) และมีเอกสารกับดัก DFC สี่ข้อ

ที่สำคัญกว่า: การมีสองฉบับคือเหตุที่บั๊กสามตัวรอดมาได้ทั้งที่ repo แก้ไปแล้ว
ตอนนี้ใช้ฉบับเดียว — ฉบับที่ sync มาจาก `main` ใน cell 0.5


In [ ]:
# ไม่มีอะไรให้เขียนแล้ว — สคริปต์มาจาก repo ที่ sync ใน cell 0.5
print('using', COMPILE_SCRIPT)
print(subprocess.check_output(['wc', '-l', str(COMPILE_SCRIPT)], text=True).strip(), 'lines')

# END_NODES ใน cell config เก็บไว้เป็นตัวเทียบเท่านั้น — สคริปต์ derive เองจาก graph
# ถ้าอยาก override (กรณี rename แปลก ๆ) ส่ง --end-nodes เข้าไป

## 7) Run: parse → quantize → compile
⚠️ ถ้า error ที่ `nms_postprocess` (syntax ขึ้นกับเวอร์ชัน DFC) → ดู `HAR output layers` ที่ print ออกมา แล้วเติม `bbox_decoders` ใน nms_config ตาม DFC docs ของเวอร์ชันเธอ


In [ ]:
import glob, os, subprocess, sys, time

# ไฟล์ที่อันตรายคือไฟล์ที่อยู่ตรง OUT_HEF พอดี เพราะรอบนี้จะเขียนทับมัน — ไม่ใช่ .hef
# ทุกไฟล์ในโฟลเดอร์ ถ้า NET_NAME ต่างกัน (เช่นรอบก่อนเป็น s รอบนี้เป็น m) ไฟล์เก่าก็แค่
# นอนอยู่เฉย ๆ ไม่ปนกัน การเตือนรวมทุกไฟล์ทำให้แยกสองกรณีนี้ไม่ออก
print('จะเขียนไปที่:', OUT_HEF)
collision = os.path.exists(OUT_HEF)
if collision:
    print(f'  ⚠️  มีไฟล์ชื่อนี้อยู่แล้ว (แก้ไขล่าสุด {time.ctime(os.path.getmtime(OUT_HEF))}) — รอบนี้จะทับ')
others = [p for p in glob.glob(f'{WORK}/*.hef') if p != OUT_HEF]
if others:
    print('  (ไฟล์อื่นในโฟลเดอร์ ไม่เกี่ยวกับรอบนี้:',
          ', '.join(f'{os.path.basename(p)} {time.ctime(os.path.getmtime(p))}' for p in others) + ')')

# stream log ของ DFC ออกมาสด ๆ
#
# ก่อนหน้านี้ cell นี้ใช้ subprocess.run เฉย ๆ แล้ว log ของ Hailo หายไปทั้งหมด: ลูกเขียนลง
# fd ของ kernel ซึ่งไม่ได้ต่อกับ output ของ cell ใน Colab ต่างจาก `!` ที่ IPython จับมาแสดงให้
# compile กินเวลาหลายนาที การรันแบบไม่เห็นอะไรเลยแปลว่าไม่รู้ว่าคืบหน้าหรือค้าง และถ้า DFC
# บ่นอะไรก็ไม่มีทางเห็น — Popen + PIPE แล้ว print ทีละบรรทัดผ่าน sys.stdout ที่ IPython
# redirect ไว้ จึงได้ทั้ง log สดและ exit code
started = time.time()
cmd = [
    # -u ไม่ใช่ของประดับ: พอ stdout เป็น pipe (ไม่ใช่ tty) Python จะ block-buffer
    # แล้ว log ทั้งหมดจะโผล่ทีเดียวตอนจบ ซึ่งก็คือ 'ไม่เห็นอะไรเลย' เหมือนเดิม
    f'{VENV}/python', '-u', str(COMPILE_SCRIPT),
    '--onnx', ONNX_PATH, '--calib', CALIB_DIR, '--out', OUT_HEF, '--work', WORK,
    '--hw', HW_ARCH, '--net', NET_NAME, '--classes', str(NUM_CLASSES),
    '--size', str(INPUT_SIZE), '--calib-n', str(CALIB_N),
    '--scores-th', str(NMS_SCORES_TH), '--iou-th', str(NMS_IOU_TH),
    '--max-per-class', str(MAX_PER_CLASS), '--reg-len', str(REG_LENGTH),
]
print('\n$', ' '.join(cmd), '\n', flush=True)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print(f'\n[compile] exit {rc} · {time.time() - started:.0f}s', flush=True)

assert rc == 0, f'compile ล้ม (exit {rc}) — ดู log ข้างบน อย่าใช้ .hef ที่อยู่ใน {WORK}'

# exit 0 อย่างเดียวยังไม่พอ: ต้องมีไฟล์ที่ถูกเขียนหลังเวลาที่เริ่มรันรอบนี้จริง ๆ
assert os.path.exists(OUT_HEF), f'compile จบด้วย exit 0 แต่ไม่มีไฟล์ที่ {OUT_HEF}'
assert os.path.getmtime(OUT_HEF) > started, (
    f'{OUT_HEF} มีอยู่ แต่ไม่ได้ถูกเขียนใหม่ในรอบนี้ (mtime '
    f'{time.ctime(os.path.getmtime(OUT_HEF))}) — ถือว่าไม่สำเร็จ'
)
HEF_PATH = OUT_HEF
print('HEF (สร้างรอบนี้):', HEF_PATH, f'{os.path.getsize(HEF_PATH)/1e6:.1f} MB')

## 8) ✅ Verify HEF ตรง edge contract (จับ silent failure ก่อน deploy)
input ต้อง **UINT8 640×640×3**, output ต้องเป็น **NMS**.
`hailo_platform` (HailoRT) อาจไม่อยู่ใน venv — ถ้าไม่มี ให้ verify บน Pi ด้วย `hailortcli parse-hef`


In [ ]:
%%writefile /content/verify_hef.py
import sys, glob
work = sys.argv[1]; size = int(sys.argv[2])
hef_path = sorted(glob.glob(f'{work}/*.hef'))[0]; print('HEF:', hef_path)
try:
    from hailo_platform import HEF
    hef = HEF(hef_path)
    i = hef.get_input_vstream_infos()[0]; outs = hef.get_output_vstream_infos()
    print('input  :', i.name, i.shape, i.format.type)
    print('outputs:', [(o.name, str(o.format.order)) for o in outs])
    assert 'UINT8' in str(i.format.type), 'input ไม่ใช่ uint8 (เช็ค normalization)'
    assert list(i.shape[:2]) == [size,size], f'input shape {i.shape}'
    assert any('NMS' in str(o.format.order).upper() for o in outs), 'output ไม่ใช่ NMS'
    print('✅ contract ผ่าน (uint8 + shape + NMS). คลาส=2 ยืนยันบน Pi)')
except ModuleNotFoundError:
    print('⚠️ hailo_platform ไม่มีใน venv — verify บน Pi: hailortcli parse-hef', hef_path)


In [ ]:
!/content/hailo_venv/bin/python /content/verify_hef.py {WORK} {INPUT_SIZE}


## 9) Eval ด้วย emulator (วัด int8 mAP ไม่ต้องมีชิป) — optional
เทียบ fp32 vs int8 ตาม gate: mAP50 drop ≤3%, recall drop ≤5%. ต้องมี eval set (รูป+label) + model-zoo whl


In [ ]:
# ต้องติด model-zoo whl ก่อน (cell 4) + มี eval set
# VENV='/content/hailo_venv/bin'
# !{VENV}/hailomz eval --hw-arch {HW_ARCH} --target emulator \
#    --har {WORK}/{NET_NAME}_quantized.har {NET_NAME}   # --data-path /content/eval
print('eval optional — หรือใช้ event-GT ของ Loom วัด count-error บน vid1/2/3 (ตรงงานจริงกว่า)')


## 10) meta.yaml + download HEF
runtime ฝั่ง Pi ปฏิเสธถ้า sha256 ไม่ตรง


In [ ]:
import hashlib, time, yaml
# HEF_PATH comes from the compile cell, which proved the file is from this run.
hef_path = HEF_PATH
sha = hashlib.sha256(open(hef_path,'rb').read()).hexdigest()
meta = {
    'model': NET_NAME,
    'version': f'colab-{time.strftime("%Y-%m-%d")}',
    'source_commit': SOURCE_COMMIT,      # sha ตอน export onnx (config cell)
    'compile_commit': COMPILE_COMMIT,    # sha ที่ compile รอบนี้ (cell 0.5)
    'dataset_hash': DATASET_HASH,
    'hailort_version': '4.20.0',
    'input_shape': [INPUT_SIZE, INPUT_SIZE, 3],
    'class_names': CLASS_NAMES,
    'end_node_names': END_NODES,
    'quantization': {'mode': 'int8', 'calibration_size': CALIB_N, 'per_channel': True},
    'nms': {'scores_th': NMS_SCORES_TH, 'iou_th': NMS_IOU_TH, 'max_per_class': MAX_PER_CLASS, 'classes': NUM_CLASSES},
    'accuracy': {'fp32': {'mAP50': None, 'recall': None}, 'int8': {'mAP50': None, 'recall': None}},  # เติมจาก eval
    'gates_passed': False,
    'sha256': sha,
}
mp = hef_path + '.meta.yaml'
with open(mp,'w') as f: yaml.safe_dump(meta, f, sort_keys=False, allow_unicode=True)
print(open(mp).read())
from google.colab import files; files.download(hef_path); files.download(mp)


## 11) ✅ ทดสอบบน Pi (สำคัญสุด — พิสูจน์เวอร์ชันตรง)
```bash
scp yolov11s_sack.hef yolov11s_sack.hef.meta.yaml edge-rpi:~/
hailortcli fw-control identify     # ยืนยัน HAILO8L + HailoRT 4.20.0
hailortcli run yolov11s_sack.hef   # โหลด+รันได้ = เวอร์ชันเข้ากันได้
```
error `invalid compiled format` = DFC/HailoRT ไม่ match → ถอยมา DFC 3.30.0 + model-zoo 2.14.0 แล้ว compile ใหม่

---
_Loom Oracle (AI) — end-nodes/input/2-class verified จาก best.onnx จริง (2026-06-23). จุดที่อาจต้องปรับตามเวอร์ชัน DFC: `nms_postprocess` syntax + bbox_decoders (ดู HAR output layers ที่ print ใน section 7)._
